In [1]:
import pandas as pd

# Load the dataset
file_path = '/content/Carbon_FootPrint_Annual.csv'
df = pd.read_csv(file_path)

# Display the first 5 rows of the DataFrame
display(df.head())

### Initial Data Inspection
Let's check the DataFrame's information to understand data types and non-null values, and then view descriptive statistics.

In [2]:
# Get a concise summary of the DataFrame
df.info()

# Display descriptive statistics
display(df.describe())

import pandas as pd
import numpy as np

# Load the ANNUAL dataset
df = pd.read_csv('Carbon_FootPrint_Annual.csv')

# Display basic information about the dataset
print('Shape:', df.shape)
print('\nColumn names:', df.columns.tolist())
print('\nFirst 5 rows:')
display(df.head())
print('\nData types:')
print(df.dtypes)


In [3]:
# List of columns to clean
columns_to_clean = [
    'Personal_Vehicle_Km_Annual',
    'Public_Vehicle_Km_Annual',
    'Electricity_Kwh_Annual',
    'Water_Usage_Liters_Annual',
    'Carbon_Footprint_Kg_Annual'
]

# Replace negative values with 0 in the specified columns
for col in columns_to_clean:
    df[col] = df[col].apply(lambda x: max(x, 0))

# Display descriptive statistics again to verify the changes
display(df[columns_to_clean].describe())

### Exploratory Data Analysis (EDA): Visualizations
Now that the data has been cleaned, let's perform some exploratory data analysis to understand the distributions of features and their relationships, especially with the target variable `Carbon_Footprint_Kg_Annual`.

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style for the plots
sns.set_style("whitegrid")

# Get numerical columns (excluding the target for now, to plot separately)
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

# Drop Carbon_Footprint_Kg_Annual from this list to plot its distribution separately later if needed
if 'Carbon_Footprint_Kg_Annual' in numerical_cols:
    numerical_cols.remove('Carbon_Footprint_Kg_Annual')

# Create histograms for all numerical features
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(3, 3, i) # Adjust subplot grid size as needed
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

### Correlation Analysis
Let's visualize the correlation matrix to understand the linear relationships between all numerical features, including our target variable `Carbon_Footprint_Kg_Annual`.

In [5]:
# Calculate the correlation matrix
correlation_matrix = df.select_dtypes(include=['float64', 'int64']).corr()

# Plot the correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numerical Features')
plt.show()

### Relationship with Target Variable (`Carbon_Footprint_Kg_Annual`)
Now, let's look at scatter plots to directly observe the relationship of each numerical feature with `Carbon_Footprint_Kg_Annual`.

In [6]:
# Create scatter plots of numerical features vs. Carbon_Footprint_Kg_Annual
plt.figure(figsize=(15, 12))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(3, 3, i) # Adjust subplot grid size as needed
    sns.scatterplot(x=df[col], y=df['Carbon_Footprint_Kg_Annual'])
    plt.title(f'{col} vs. Carbon_Footprint_Kg_Annual')
    plt.xlabel(col)
    plt.ylabel('Carbon_Footprint_Kg_Annual')
plt.tight_layout()
plt.show()

### Analyze Categorical Feature: `Diet_Type`
Finally, let's explore how the categorical feature `Diet_Type` relates to `Carbon_Footprint_Kg_Annual`.

In [8]:
# Analyze 'Diet_Type' against 'Carbon_Footprint_Kg_Annual'
plt.figure(figsize=(8, 6))
sns.boxplot(x='Diet_Type', y='Carbon_Footprint_Kg_Annual', data=df, palette='viridis', hue='Diet_Type', legend=False)
plt.title('Annual Carbon Footprint by Diet Type')
plt.xlabel('Diet Type')
plt.ylabel('Carbon_Footprint_Kg_Annual')
plt.show()

### Feature Engineering: Handling Categorical Variables
Now, let's encode the categorical `Diet_Type` column using one-hot encoding to prepare it for machine learning models.

In [9]:
# Apply one-hot encoding to 'Diet_Type'
df = pd.get_dummies(df, columns=['Diet_Type'], drop_first=True)

# Display the first few rows of the DataFrame with the new encoded columns
display(df.head())

### Model Building: Random Forest Regressor
Now, let's prepare the data for machine learning and build a Random Forest Regressor model to predict `Carbon_Footprint_Kg_Annual`.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Define features (X) and target (y)
X = df.drop('Carbon_Footprint_Kg_Annual', axis=1)
y = df['Carbon_Footprint_Kg_Annual']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training set shape: {X_train.shape}')
print(f'Testing set shape:  {X_test.shape}')


In [11]:
# Initialize the Random Forest Regressor model
# You can tune hyperparameters like n_estimators, max_depth, etc.
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# Train the model
rf_model.fit(X_train, y_train)

print("Random Forest Regressor model trained successfully!")

### Model Evaluation
Let's evaluate the performance of the Random Forest Regressor on the test set using key regression metrics.

In [13]:
# Make predictions on the test set
y_pred_rf = rf_model.predict(X_test)

# Evaluate the model
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = mse_rf**0.5  # Calculate RMSE by taking the square root of MSE
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest Regressor Performance:")
print(f"  Mean Absolute Error (MAE): {mae_rf:.4f}")
print(f"  Mean Squared Error (MSE): {mse_rf:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_rf:.4f}")
print(f"  R-squared (R2): {r2_rf:.4f}")

### Model Performance Summary

In [14]:
model_performance = pd.DataFrame({
    'Metric': ['Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)', 'Root Mean Squared Error (RMSE)', 'R-squared (R2)'],
    'Random Forest Regressor': [mae_rf, mse_rf, rmse_rf, r2_rf]
})

display(model_performance)

### Hyperparameter Tuning with GridSearchCV
Let's use GridSearchCV to find the optimal hyperparameters for our Random Forest Regressor. This will help us potentially improve the model's accuracy.

In [16]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 200],  # Number of trees in the forest
    'max_features': ['sqrt', 'log2'],  # Number of features to consider when looking for the best split
    'max_depth': [None, 10, 20, 30],  # Maximum depth of the tree
    'min_samples_split': [2, 5, 10],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4]  # Minimum number of samples required to be at a leaf node
}

# Initialize a new Random Forest Regressor instance
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid,
                           cv=3, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

print("GridSearchCV completed successfully!")

In [17]:
# Print the best parameters and the best score
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best negative MSE score: {grid_search.best_score_}")

# Get the best model
best_rf_model = grid_search.best_estimator_

# Make predictions with the best model
y_pred_best_rf = best_rf_model.predict(X_test)

# Evaluate the best model
mae_best_rf = mean_absolute_error(y_test, y_pred_best_rf)
mse_best_rf = mean_squared_error(y_test, y_pred_best_rf)
rmse_best_rf = mse_best_rf**0.5
r2_best_rf = r2_score(y_test, y_pred_best_rf)

print(f"\nRandom Forest Regressor Performance with Best Parameters:")
print(f"  Mean Absolute Error (MAE): {mae_best_rf:.4f}")
print(f"  Mean Squared Error (MSE): {mse_best_rf:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_best_rf:.4f}")
print(f"  R-squared (R2): {r2_best_rf:.4f}")

### Comparison of Model Performance (Before and After Tuning)

In [18]:
# Update the model_performance DataFrame with the tuned model's metrics
model_performance['Tuned Random Forest Regressor'] = [
    mae_best_rf, mse_best_rf, rmse_best_rf, r2_best_rf
]

display(model_performance)

### Model Building: XGBoost Regressor
Let's train an XGBoost Regressor model and evaluate its performance to compare with the Random Forest models.

In [19]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Define features (X) and target (y)
X = df.drop('Carbon_Footprint_Kg_Annual', axis=1)
y = df['Carbon_Footprint_Kg_Annual']

# Same 80/20 split as before for fair comparison
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the XGBoost Regressor model
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)

# Train the model
xgb_model.fit(X_train, y_train)

print('XGBoost Regressor model trained successfully!')


### XGBoost Model Evaluation

In [20]:
# Make predictions on the test set
y_pred_xgb = xgb_model.predict(X_test)

# Evaluate the model
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
rmse_xgb = mse_xgb**0.5
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"XGBoost Regressor Performance:")
print(f"  Mean Absolute Error (MAE): {mae_xgb:.4f}")
print(f"  Mean Squared Error (MSE): {mse_xgb:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_xgb:.4f}")
print(f"  R-squared (R2): {r2_xgb:.4f}")

### Comparison of All Model Performances

In [21]:
# Update the model_performance DataFrame with XGBoost metrics
model_performance['XGBoost Regressor'] = [
    mae_xgb, mse_xgb, rmse_xgb, r2_xgb
]

display(model_performance)

### Visual Comparison of Model Performances
Let's visualize the performance metrics of all three models to get a clear comparative view.

In [22]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")

# Prepare data for plotting
metrics = model_performance['Metric'].tolist()
rf_scores = model_performance['Random Forest Regressor'].tolist()
tuned_rf_scores = model_performance['Tuned Random Forest Regressor'].tolist()
xgb_scores = model_performance['XGBoost Regressor'].tolist()

num_metrics = len(metrics)
ind = np.arange(num_metrics)  # the x locations for the groups
width = 0.2  # the width of the bars

fig, ax = plt.subplots(figsize=(14, 7))

# Plotting bars for each model
rects1 = ax.bar(ind - width, rf_scores, width, label='Random Forest Regressor', color='skyblue')
rects2 = ax.bar(ind, tuned_rf_scores, width, label='Tuned Random Forest Regressor', color='lightcoral')
rects3 = ax.bar(ind + width, xgb_scores, width, label='XGBoost Regressor', color='lightgreen')

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_xlabel('Metrics')
ax.set_ylabel('Score')
ax.set_title('Comparison of Model Performance Metrics')
ax.set_xticks(ind)
ax.set_xticklabels(metrics, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

### Final Model Selection and Saving
Based on the comparative analysis, the **XGBoost Regressor** generally exhibits the best performance with the lowest MAE, MSE, RMSE, and the highest R-squared value.

Therefore, we will select the XGBoost Regressor as our final model. We will now save this model for future use.

In [23]:
import joblib

# Select the best model (XGBoost Regressor)
best_model = xgb_model

# Save the model to disk
model_filename = 'best_carbon_footprint_model.joblib'
joblib.dump(best_model, model_filename)

print(f'Best model (XGBoost Regressor) saved as {model_filename}')
print('Feature order:')
for i, col in enumerate(X.columns):
    print(f'  {i+1:2d}. {col}')
